# Didattica con ChatGPT

## Setup

In [ ]:
# Install a pip package in the current Jupyter kernel
import sys
!{sys.executable} -m pip install openai

In [1]:
import openai
import os

client = None
if 'OPENAI_API_KEY' in os.environ:
    client = openai.OpenAI(api_key=os.environ['OPENAI_API_KEY'])  # this is also the default, it can be omitted
    client
else:
    print("'OPENAI_API_KEY' is not set")

In [2]:
# EV: old
# def get_completion(prompt, model="gpt-3.5-turbo", temperature=0): 
#     messages = [{"role": "user", "content": prompt}]
#     response = openai.ChatCompletion.create(
#         model=model,
#         messages=messages,
#         temperature=temperature, 
#     )
#     return response.choices[0].message["content"]

def ev_completion(client, model_par: str, prompt_par: str):
    """
    https://stackoverflow.com/questions/75774873/openai-chatgpt-gpt-3-5-api-error-this-is-a-chat-model-and-not-supported-in-t
    """
    try:
        completion_pyd = client.chat.completions.create( # Change the method name
            model = model_par,
            messages = [ # Change the prompt parameter to messages parameter
                {'role': 'user', 'content': prompt_par}
            ],
            temperature = 0  
        )
    except Exception as exc:
        print(exc)
        return None
    
    completion_d = completion_pyd.model_dump()
    reply0_txt = completion_d['choices'][0]['message']['content']
    return reply0_txt, completion_d, completion_pyd


def get_completion(prompt, model="gpt-4o"):
# def get_completion(prompt, model="gpt-3.5-turbo"):
    global client
    ret = ev_completion(client, model, prompt)
    return ret[0]

In [ ]:
def generate_prompt(materia1: str, materia2 : str, lista_temi, lista_disturbi,
    nr_domande = 15, nr_risposte = 5):

    temi = ", ".join(lista_temi)

    prompt =  f"""
    genera {nr_domande} domande di {materia1} nell'ambito di {materia2} sui temi seguenti: {temi}.

    Per ogni domanda genera {nr_risposte} risposte, una sola corretta. 
    Nelle risposte includi sempre l’opzione “nessuna è corretta”.
    Nelle risposte includi sempre l’opzione “altro, specificare”.
    Per ogni domanda genera anche una variante semplificata per studenti con i seguenti disturbi {lista_disturbi}.
    Il formato deve essere JSON, con le seguenti chiavi:
    - domanda
    - risposte_l, per la lista delle risposte
    - risposta corretta
    """
    return prompt

materia1 = "Italiano"
materia2 = "letteratura"
lista_temi = ["Manzoni", "Leopardi"]
nr_domande = 10
nr_risposte = 5
lista_disturbi = ["dislessia"]


prompt =  generate_prompt(materia1, materia2, lista_temi, lista_disturbi=lista_disturbi)

print(prompt)


    genera 15 domande di Italiano nell'ambito di letteratura sui temi seguenti: Manzoni, Leopardi.

    Per ogni domanda genera 5 risposte, una sola corretta. 
    Nelle risposte includi sempre l’opzione “nessuna è corretta”.
    Nelle risposte includi sempre l’opzione “altro, specificare”.
    Per ogni domanda genera anche una variante semplificata per studenti con i seguenti disturbi ['dislessia'].
    Il formato deve essere JSON, con le seguenti chiavi:
    - domanda
    - risposte_l, per la lista delle risposte
    - risposta corretta
    


In [4]:
response = get_completion(prompt)
print(response)

Ecco un esempio di domande di letteratura sui temi di Manzoni e Leopardi, formattate in JSON:

```json
[
    {
        "domanda": "Qual è il tema principale de 'I Promessi Sposi' di Alessandro Manzoni?",
        "risposte_l": [
            "L'amore contrastato",
            "La giustizia divina",
            "La peste",
            "La lotta di classe",
            "nessuna è corretta",
            "altro, specificare"
        ],
        "risposta corretta": "La giustizia divina"
    },
    {
        "domanda": "Qual è il tema principale de 'I Promessi Sposi' di Alessandro Manzoni? (semplificata)",
        "risposte_l": [
            "L'amore contrastato",
            "La giustizia divina",
            "La peste",
            "La lotta di classe",
            "nessuna è corretta",
            "altro, specificare"
        ],
        "risposta corretta": "La giustizia divina"
    },
    {
        "domanda": "In quale anno è stato pubblicato 'I Promessi Sposi' nella sua versione definitiv

In [5]:
import json

domande = json.loads(response)
domande

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
import random

def generate_domande(domande):

    ret = ""
    RISPOSTA_CORRETTA_KEY = "risposta corretta"

    for nr, domanda_d in enumerate(domande, start=1):
        domanda = domanda_d['domanda']
        # print(f"{nr:0>2} {domanda}")
        ret += f"{nr:0>2} {domanda}"+"\n"
        # random.shuffle(domanda_d['risposte_l']) # mette risp. corretta sempre stesso post
        for risposta in domanda_d['risposte_l']:
            if "corretta" in risposta.lower() and "risposta" in risposta.lower():
                continue # non stamparla
            ret += f"o {risposta}"+"\n"
        if RISPOSTA_CORRETTA_KEY in domanda_d:
            risposta_corretta = domanda_d[RISPOSTA_CORRETTA_KEY]
            ret += "### corretta: "+risposta_corretta+"\n"
            #.split(":")[1].strip()
            # print(risposta_corretta)
            # pass
        else:
            print(domanda_d.keys())

    return ret

In [ ]:
ret = generate_domande(domande)
print(ret)


In [ ]:
prompt = f"""
format the text delimited by ### in an HTML table
put both the questions and each answer on its own line.
Make the table higly readable for font colours
###
{ret}
###
"""

html_table = get_completion(prompt)
html_table

In [ ]:
from IPython.display import display, Markdown, Latex, HTML, JSON
display(HTML(html_table))

## Traduzione

ChatGPT è addestrato con fonti in molte lingue. Ciò dà al modello la capacità di eseguire traduzioni. Ecco alcuni esempi di come utilizzare questa funzionalità.

In [ ]:
prompt = f"""
Traduci il testo Inglese seguente in Italiano: \ 
```Hi, I would like to order a beer```
"""
response = get_completion(prompt)
print(response)

In [ ]:
prompt = f"""
dimmi in che linguaggio è il testo delimitato da ```: 
```Combien coûte le lampadaire?```
"""
response = get_completion(prompt)
print(response)

In [ ]:
prompt = f"""
traduci il testo delimitato da ``` in Francese, Spagnolo e English pirate: \
```I want to order a basketball```
"""
response = get_completion(prompt)
print(response)

In [ ]:
prompt = f"""
Traduci il testo seguente in Italiano, sia in linguggio fromale sia colloquiale: 
'Would you like to order a pillow?'
"""
response = get_completion(prompt)
print(response)

### Traduttore universale  
Immagina di essere responsabile dell'IT presso una grande azienda multinazionale di e-commerce. Gli utenti ti inviano messaggi relativi a problemi IT in tutte le loro lingue native. Il tuo personale proviene da tutto il mondo e parla solo la propria lingua madre. Hai bisogno di un traduttore universale!

In [ ]:
user_messages = [
  "La performance du système est plus lente que d'habitude.",  # System performance is slower than normal         
  "Mi monitor tiene píxeles que no se iluminan.",              # My monitor has pixels that are not lighting
  "Il mio mouse non funziona",                                 # My mouse is not working
  "Mój klawisz Ctrl jest zepsuty",                             # My keyboard has a broken control key
  "我的屏幕在闪烁"                                               # My screen is flashing
] 

In [ ]:
for issue in user_messages:
    prompt = f"Tell me what language this is: ```{issue}```"
    lang = get_completion(prompt)
    print(f"\n\nOriginal message ({lang}): {issue}")

    prompt = f"""
    Translate the following  text to English \
    and Korean: ```{issue}```
    """
    response = get_completion(prompt)
    print(response)

## Trasformazione del tono  
La scrittura può variare in base al pubblico a cui si rivolge. ChatGPT può produrre toni diversi.

In [ ]:
prompt = f"""
Traduci il testo seguente dallo slang in una lettera commerciale: 

'Ehi fratello, dai un'occhiata alle specifiche di questa cavolo di lampada da tavolo,
che io mi sono proprio rotto, non ce la faccio.
E ricordate che aspetto il dinero, nun ce dormi sopra'
"""
response = get_completion(prompt)
# print(response)
print("\n".join(response.split(".")))

## Conversioni di formato  
ChatGPT può tradurre tra formati "tecnici". Il prompt dovrebbe descrivere i formati di input e output.

In [ ]:
data_json = { "resturant employees" :[ 
    {"name":"Shyam", "email":"shyamjaiswal@gmail.com"},
    {"name":"Bob", "email":"bob32@gmail.com"},
    {"name":"Jai", "email":"jai87@gmail.com"}
]}

prompt = f"""
Translate the following python dictionary from JSON to an HTML \
table with column headers and title: {data_json}
"""
response = get_completion(prompt)
print(response)

In [ ]:
from IPython.display import display, Markdown, Latex, HTML, JSON
display(HTML(response))

## Controllo di ortografia e grammatica  

Ecco alcuni esempi di problemi grammaticali e ortografici comuni e la risposta del LLM.

Per segnalare al LLM che desideri che corregga il tuo testo, istruisci il modello a "correggere" o "correggere e correggere".

In [ ]:
text = [ 
  "The girl with the black and white puppies have a ball.",  # The girl has a ball.
  "Yolanda has her notebook.", # ok
  "Its going to be a long day. Does the car need it’s oil changed?",  # Homonyms
  "Their goes my freedom. There going to bring they’re suitcases.",  # Homonyms
  "Your going to need you’re notebook.",  # Homonyms
  "That medicine effects my ability to sleep. Have you heard of the butterfly affect?", # Homonyms
  "This phrase is to cherck chatGPT for speling abilitty"  # spelling
]

text = [ 
  "Ieri ho andato alla mare",
  "Quel collega è troppo chiacchieroni.",
]



for t in text:
    prompt = f"""Controlla e correggi il testo seguente, e scrivi la versione corretta.
    Se non trovi errori, dì semplicemente "Nessun errore trovato".
    Non utilizzare segni di punteggiatura attorno al testo:
    ```{t}```"""
    response = get_completion(prompt)
    print(response)

In [ ]:
responses_l = []

for t in text:
    prompt = f"""
    Rileggi e correggi il testo delimitato da ```.
    se il testo contiene errori
    - scrivere sia la versione originale che quella corretta, ciascuna sulla propria riga.
    - anteporre alla versione originale <originale>,
    - anteporre alla versione corretta <corretta>
    - scrivere una terza riga che spieghi l'errore, con il prefisso <spiegazione>
    Se il testo non contiene errori:
    - scrivere il testo su una riga
    - ha preceduto il testo con <nessun errore>
    
    Non utilizzare simboli di punteggiatura o virgolette attorno al testo.
    
    Il testo è questo:
    ```{t}```"""
    response = get_completion(prompt)
    responses_l.append(response)

In [ ]:
for response in responses_l:
    print("-"*4+"\n"+response+"\n")

In [ ]:
responses_l = []

for t in text:
    prompt = f"""
    Rileggi e correggi il testo delimitato da ```.
    L'output deve essere in formato JSON.
    Le chiavi dovrebbero essere le seguenti:
    - "corretto", booleano
    - "originale": il testo invariato
    - "corretto": presente solo quando il testo contiene errori
    - "spiegazione": presente solo quando il testo contiene errori

    Non utilizzare simboli di punteggiatura o virgolette attorno al testo.

    Il testo è questo:
    ```{t}```"""
    response = get_completion(prompt)
    responses_l.append(response)

In [ ]:
for response in responses_l:
    print("-"*4+"\n"+response+"\n")

In [ ]:
text = f"""
Got this for my daughter for her birthday cuz she keeps taking \
mine from my room.  Yes, adults also like pandas too.  She takes \
it everywhere with her, and it's super soft and cute.  One of the \
ears is a bit lower than the other, and I don't think that was \
designed to be asymmetrical. It's a bit small for what I paid for it \
though. I think there might be other options that are bigger for \
the same price.  It arrived a day earlier than expected, so I got \
to play with it myself before I gave it to my daughter.
"""

text = f"""
L'ho prendetti per mio figlia per il suo compleanno perché continua a prendere \
il mio dalla mia stanza. Sì, anche agli adulti ci piacciono i panda. Lei lo prende \
e lo portano ovunque con lei, ed è super morbido e caruccia. \
Le recchie sono un po' più basse delle altre, e non credo che sarebbe \
progettato per essere asimmetrico. È un po' piccolo per quello che l'ho pagato\
È arrivato un giorno prima del previsto, quindi ho ricevuto \
per giocarcine io stesso prima di darlo a mia figlia."""


prompt = f"proofread and correct this review: ```{text}```"
response = get_completion(prompt)
response = "\n".join(response.split("."))
print(response)

In [ ]:
# Install a pip package in the current Jupyter kernel
import sys
!{sys.executable} -m pip install redlines

In [ ]:
from redlines import Redlines

diff = Redlines(text,response)
display(Markdown(diff.output_markdown))

In [ ]:
prompt = f"""
rileggi e correggi questa recensione. Rendila più avvincente.
Assicurati che segua la guida di stile APA e sia rivolto a un lettore avanzato.
Output in formato markdown.
Testo: ```{text}```
"""
response = get_completion(prompt)
display(Markdown(response))

# Fine  